# 3 · Supervised Fine-Tuning

In [1]:
import json
from pathlib import Path

from datasets import load_dataset
from tokenizers import Tokenizer
from transformers import PreTrainedTokenizerFast

DATASET = "Pondsiders/tinystories-gpt4-instruct"
CONTEXT = 512
ARTIFACTS = Path("artifacts")

/Users/jefferyharrell/Pondside/Workshop/Projects/lil-transformy-2/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
raw = json.loads((ARTIFACTS / "tokenizer.json").read_text())

for entry in raw["added_tokens"]:
    if entry["content"] == "<|reserved_3|>":
        entry["content"] = "<|pad|>"
vocab = raw["model"]["vocab"]
if "<|reserved_3|>" in vocab:
    vocab["<|pad|>"] = vocab.pop("<|reserved_3|>")

tokenizer = PreTrainedTokenizerFast(
    tokenizer_object=Tokenizer.from_str(json.dumps(raw)),
    bos_token="<|endoftext|>",
    eos_token="<|im_end|>",
    pad_token="<|pad|>",
)

for name in ["<|endoftext|>", "<|im_start|>", "<|im_end|>", "<|pad|>"]:
    print(f"{tokenizer.convert_tokens_to_ids(name):4d}  {name}")

   0  <|endoftext|>
   1  <|im_start|>
   2  <|im_end|>
   3  <|pad|>


In [3]:
CHAT_TEMPLATE = (
    "{{ '<|endoftext|>' }}"
    "{% for message in messages %}"
    "{{ '<|im_start|>' + message['role'] + '\n' + message['content'] + '<|im_end|>' + '\n' }}"
    "{% endfor %}"
    "{% if add_generation_prompt %}{{ '<|im_start|>assistant\n' }}{% endif %}"
)
tokenizer.chat_template = CHAT_TEMPLATE

messages = [{"role": "user", "content": "Tell me a story about a duck named Pondside."}]
print(tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False))

<|endoftext|><|im_start|>user
Tell me a story about a duck named Pondside.<|im_end|>
<|im_start|>assistant



In [4]:
dataset = load_dataset(DATASET)
print(dataset)
print()
row = dataset["train"][0]
print(row["prompt"])
print(row["story"][:80] + "...")

DatasetDict({
    train: Dataset({
        features: ['prompt', 'story', 'source_id', 'template_id', 'names', 'kind', 'all_names'],
        num_rows: 50000
    })
    validation: Dataset({
        features: ['prompt', 'story', 'source_id', 'template_id', 'names', 'kind', 'all_names'],
        num_rows: 1000
    })
})

I'd like a story about a boy named Tim, please.
One day, a little boy named Tim was eager to play outside. He saw that the sun w...


In [5]:
BOS = tokenizer.convert_tokens_to_ids("<|endoftext|>")
IM_START = tokenizer.convert_tokens_to_ids("<|im_start|>")
IM_END = tokenizer.convert_tokens_to_ids("<|im_end|>")
PAD = tokenizer.convert_tokens_to_ids("<|pad|>")


def encode(text):
    return tokenizer.encode(text, add_special_tokens=False)


def assemble(prompt, story):
    request = (
        [BOS, IM_START] + encode("user\n" + prompt) + [IM_END]
        + encode("\n") + [IM_START] + encode("assistant\n")
    )
    response = encode(story) + [IM_END]
    return request, response


request, response = assemble(row["prompt"], row["story"])
print(tokenizer.decode(request), "…")
print(f"request {len(request)} tokens, response {len(response)} tokens")

<|endoftext|><|im_start|>user
I'd like a story about a boy named Tim, please.<|im_end|>
<|im_start|>assistant
 …
request 25 tokens, response 169 tokens


In [6]:
# Training must see exactly what inference will see: the request half of every
# training example has to match the chat template's rendering token for token.
mismatches = 0
for pair in dataset["validation"]:
    request, _ = assemble(pair["prompt"], pair["story"])
    templated = tokenizer.apply_chat_template(
        [{"role": "user", "content": pair["prompt"]}],
        add_generation_prompt=True,
    )["input_ids"]
    if request != templated:
        mismatches += 1
print(f"hand assembly vs chat template: {mismatches} mismatches in {len(dataset['validation'])} pairs")

hand assembly vs chat template: 0 mismatches in 1000 pairs


In [7]:
lengths = []
for pair in dataset["train"]:
    request, response = assemble(pair["prompt"], pair["story"])
    lengths.append(len(request) + len(response))

import numpy as np
lengths = np.array(lengths)
print(f"median {int(np.median(lengths))} tokens · p95 {int(np.percentile(lengths, 95))} · max {lengths.max()}")
print(f"over {CONTEXT}: {(lengths > CONTEXT).sum():,} of {len(lengths):,} ({100 * (lengths > CONTEXT).mean():.2f}%)")

median 204 tokens · p95 285 · max 1020
over 512: 60 of 50,000 (0.12%)


In [8]:
import numpy as np

IGNORE = -100


def build_example(pair):
    request, response = assemble(pair["prompt"], pair["story"])
    length = len(request) + len(response)
    if length > CONTEXT:
        return None
    pad = CONTEXT - length
    input_ids = request + response + [PAD] * pad
    labels = [IGNORE] * len(request) + response + [IGNORE] * pad
    attention_mask = [1] * length + [0] * pad
    return input_ids, labels, attention_mask


def build_split(split):
    examples = [build_example(pair) for pair in split]
    kept = [e for e in examples if e is not None]
    ids, labels, mask = (np.array(t, dtype=np.int16) for t in zip(*kept))
    print(f"{len(kept):,} examples kept, {len(examples) - len(kept):,} dropped for length")
    return ids, labels, mask


train_ids, train_labels, train_mask = build_split(dataset["train"])
valid_ids, valid_labels, valid_mask = build_split(dataset["validation"])

49,940 examples kept, 60 dropped for length
997 examples kept, 3 dropped for length


In [9]:
# The pad token and -100 wear different hats but must agree: anywhere the
# input is padding, the label must be IGNORE, or the model learns to predict
# nothing and the loss looks wonderful while the model gets worse.
assert (train_labels[train_ids == PAD] == IGNORE).all()
assert (valid_labels[valid_ids == PAD] == IGNORE).all()

# And the graded region must be exactly the response: story plus its <|im_end|>.
graded = train_labels[0][train_labels[0] != IGNORE].astype(np.int64)
print(tokenizer.decode(graded)[:120] + " …")
print(f"…ends with: {tokenizer.convert_ids_to_tokens([int(graded[-1])])}")

One day, a little boy named Tim was eager to play outside. He saw that the sun was shining and the snow was melting. He  …
…ends with: ['<|im_end|>']


In [10]:
example = train_ids[0].astype(np.int64)
markers = ["·" if label == IGNORE else "█" for label in train_labels[0]]
tokens = tokenizer.convert_ids_to_tokens(example)
print("".join(markers[:80]))
print()
for token, marker in list(zip(tokens, markers))[:24]:
    print(f"  {marker}  {token}")

·························███████████████████████████████████████████████████████

  ·  <|endoftext|>
  ·  <|im_start|>
  ·  us
  ·  er
  ·  Ċ
  ·  I
  ·  'd
  ·  Ġlike
  ·  Ġa
  ·  Ġstory
  ·  Ġabout
  ·  Ġa
  ·  Ġboy
  ·  Ġnamed
  ·  ĠTim
  ·  ,
  ·  Ġplease
  ·  .
  ·  <|im_end|>
  ·  Ċ
  ·  <|im_start|>
  ·  ass
  ·  ist
  ·  ant


In [11]:
total = train_mask.sum()
graded = (train_labels != IGNORE).sum()
padding = (train_ids == PAD).sum()
print(f"tokens in play: {total:,}")
print(f"graded: {graded:,} ({100 * graded / total:.0f}% of real tokens)")
print(f"padding: {padding:,} ({100 * padding / train_ids.size:.0f}% of all positions)")

tokens in play: 10,493,205
graded: 9,367,757 (89% of real tokens)
padding: 15,076,075 (59% of all positions)


In [ ]:
import json
from contextlib import nullcontext

import pendulum
import torch
import transformers
from transformers import LlamaForCausalLM

BATCH_SIZE = 64
PEAK_LR = 6e-5
WARMUP_STEPS = 50
TRUNK_EPOCHS = 3
ANNEAL_STEPS = 100
GRAD_CLIP = 1.0
SEED = 20260831

STEPS_PER_EPOCH = len(train_ids) // BATCH_SIZE
TRUNK_STEPS = TRUNK_EPOCHS * STEPS_PER_EPOCH
PAIRS_PER_EPOCH = STEPS_PER_EPOCH * BATCH_SIZE

# Checkpoints peel off the trunk when this many pairs have been seen. The last
# two are one full epoch and the end of the trunk.
MILESTONES = [1_000, 5_000, 20_000, PAIRS_PER_EPOCH, TRUNK_EPOCHS * PAIRS_PER_EPOCH]

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
torch.manual_seed(SEED)

# Mixed precision, as in pretraining: the weights stay fp32 (the optimizer
# needs the precision to feel small updates), the forward and backward passes
# run in bf16.
amp = nullcontext() if device == "cpu" else torch.autocast(device, dtype=torch.bfloat16)

model = LlamaForCausalLM.from_pretrained(ARTIFACTS / "lil-transformy-2").to(device)
model.train()

optimizer = torch.optim.AdamW(
    model.parameters(), lr=PEAK_LR, betas=(0.9, 0.95), weight_decay=0.1,
    fused=(device == "cuda"),
)

# Every run gets its own directory, named by when it started (house time), and
# a config.json saying what it was and what it ran on.
RUN = ARTIFACTS / "sft" / pendulum.now("America/Los_Angeles").format("YYYY-MM-DD_HHmm")
RUN.mkdir(parents=True, exist_ok=True)
config = {
    "run": RUN.name,
    "started": pendulum.now("America/Los_Angeles").to_datetime_string(),
    "device": device,
    "gpu": torch.cuda.get_device_name(0) if device == "cuda" else None,
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "base_model": str(ARTIFACTS / "lil-transformy-2"),
    "dataset": DATASET,
    "context": CONTEXT,
    "precision": "bf16 autocast over fp32 masters",
    "batch_size": BATCH_SIZE,
    "peak_lr": PEAK_LR,
    "warmup_steps": WARMUP_STEPS,
    "trunk_epochs": TRUNK_EPOCHS,
    "trunk_steps": TRUNK_STEPS,
    "anneal_steps": ANNEAL_STEPS,
    "grad_clip": GRAD_CLIP,
    "seed": SEED,
    "milestones": MILESTONES,
}
(RUN / "config.json").write_text(json.dumps(config, indent=2) + "\n")

print(f"run {RUN}")
print(f"{config['gpu'] or device} · {STEPS_PER_EPOCH} steps/epoch · trunk {TRUNK_STEPS} steps · "
      f"milestones at pairs {MILESTONES}")

In [ ]:
# Warmup-Stable-Decay: warm up, then hold the trunk FLAT. There is no decay
# here — every checkpoint peeled off the trunk gets its own short anneal
# instead, so one run can father many finished models.
def trunk_lr(step):
    if step < WARMUP_STEPS:
        return PEAK_LR * (step + 1) / WARMUP_STEPS
    return PEAK_LR


# Pad to the batch, not to the context. Every example was padded out to 512
# positions on disk; a batch only needs to be as wide as its longest member.
def to_batch(ids, mask, labels):
    width = int(mask.sum(axis=1).max())
    return {
        "input_ids": torch.from_numpy(ids[:, :width].astype(np.int64)).to(device),
        "attention_mask": torch.from_numpy(mask[:, :width].astype(np.int64)).to(device),
        "labels": torch.from_numpy(labels[:, :width].astype(np.int64)).to(device),
    }


def batches(epoch_seed):
    order = np.random.default_rng(epoch_seed).permutation(len(train_ids))
    for start in range(0, len(order) - BATCH_SIZE + 1, BATCH_SIZE):
        rows = order[start : start + BATCH_SIZE]
        yield to_batch(train_ids[rows], train_mask[rows], train_labels[rows])


def train_step(net, opt, batch, lr):
    for group in opt.param_groups:
        group["lr"] = lr
    with amp:
        loss = net(**batch).loss
    opt.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(net.parameters(), GRAD_CLIP)
    opt.step()
    return loss


@torch.no_grad()
def validation_loss(net, n_batches=8):
    net.eval()
    total = 0.0
    for start in range(0, n_batches * BATCH_SIZE, BATCH_SIZE):
        rows = slice(start, start + BATCH_SIZE)
        with amp:
            total += net(**to_batch(valid_ids[rows], valid_mask[rows], valid_labels[rows])).loss.item()
    net.train()
    return total / n_batches

In [ ]:
# The trunk. Three passes over the pairs at a flat learning rate. When the
# count of pairs seen crosses a milestone, the fp32 weights and the optimizer
# state are saved as they are — no cooldown yet, that comes next.
started = pendulum.now()
print(f"validation loss before training: {validation_loss(model):.4f}")

peeled = []
pending = list(MILESTONES)
step = 0
pairs_seen = 0
history = []

for epoch in range(TRUNK_EPOCHS):
    for batch in batches(epoch_seed=SEED + epoch):
        loss = train_step(model, optimizer, batch, trunk_lr(step))
        step += 1
        pairs_seen += BATCH_SIZE

        if step % 100 == 0 or step == TRUNK_STEPS:
            valid = validation_loss(model)
            history.append((step, pairs_seen, loss.item(), valid))
            print(f"step {step:5d} · pairs {pairs_seen:7,d} · lr {trunk_lr(step - 1):.1e} · "
                  f"train {loss.item():.4f} · valid {valid:.4f}")

        if pending and pairs_seen >= pending[0]:
            pending.pop(0)
            where = RUN / "trunk" / f"pairs-{pairs_seen:06d}"
            model.save_pretrained(where)
            torch.save(optimizer.state_dict(), where / "optimizer.pt")
            peeled.append(where)
            print(f"  peeled {where.relative_to(RUN)} · valid {validation_loss(model):.4f}")

elapsed = pendulum.now() - started
config["trunk_seconds"] = round(elapsed.total_seconds(), 1)
(RUN / "config.json").write_text(json.dumps(config, indent=2) + "\n")
print(f"trunk done in {elapsed.in_words()} · {TRUNK_STEPS / elapsed.total_seconds():.1f} steps/s")

In [ ]:
# The anneals. Each peeled checkpoint picks up exactly where the trunk left
# it — same weights, same optimizer moments — and runs the learning rate
# linearly down to zero. The result is saved in bf16 with the tokenizer and
# chat template riding along: a complete model, nothing will train from it.
def anneal(trunk_dir, seed):
    net = LlamaForCausalLM.from_pretrained(trunk_dir).to(device)
    net.train()
    opt = torch.optim.AdamW(
        net.parameters(), lr=PEAK_LR, betas=(0.9, 0.95), weight_decay=0.1,
        fused=(device == "cuda"),
    )
    opt.load_state_dict(torch.load(trunk_dir / "optimizer.pt", map_location=device))
    for i, batch in enumerate(batches(epoch_seed=seed)):
        if i >= ANNEAL_STEPS:
            break
        train_step(net, opt, batch, PEAK_LR * (ANNEAL_STEPS - i) / ANNEAL_STEPS)
    return net


for i, trunk_dir in enumerate(peeled):
    net = anneal(trunk_dir, seed=SEED + 1_000 + i)
    valid = validation_loss(net)
    out = RUN / "annealed" / trunk_dir.name
    net.to(torch.bfloat16).save_pretrained(out)
    tokenizer.save_pretrained(out)
    print(f"{out.relative_to(RUN)} · valid {valid:.4f}")

In [ ]:
# The moment of truth: a request the model has never seen, in words no
# template used. House recipe — temperature 0.7, top_p 0.95.
final = LlamaForCausalLM.from_pretrained(RUN / "annealed" / peeled[-1].name).to(device)
final.eval()

request = "Could you tell me a story about a duck named Pondside who finds a shiny stone?"
prompt = tokenizer.apply_chat_template(
    [{"role": "user", "content": request}], add_generation_prompt=True, return_tensors="pt",
)["input_ids"].to(device)

torch.manual_seed(SEED)
with torch.no_grad():
    out = final.generate(
        prompt, max_new_tokens=300, do_sample=True, temperature=0.7, top_p=0.95,
        eos_token_id=IM_END, pad_token_id=PAD,
    )
story = tokenizer.decode(out[0][prompt.shape[1]:], skip_special_tokens=True)
print(request)
print()
print(story)
print()
print(f"run {RUN}")